# OAKG — FLARE evaluation (all FLARE analyses in one place)

Corresponds to the FLARE-Evaluation note: how **FLARE22 / FLARE23 / FLARE24** are each
used in OAKG, kept at precise evaluation levels. Runs on the **committed artifacts**
(`benchmark/`, `results/`) — no raw masks needed.

1. Dataset roles & provenance  2. FLARE22 UNOBSERVED-tumor observability  
3. FLARE22's contribution to the main benchmark  4. FLARE23 slice-level tumor (exploratory)  
5. Cancer-query examples  6. FLARE24 (audit pending)


In [1]:
import json, pandas as pd, numpy as np
from pathlib import Path
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 60)
# find repo root (dir containing benchmark/ and results/)
ROOT = Path.cwd()
while not ((ROOT/'benchmark').exists() and (ROOT/'results').exists()) and ROOT != ROOT.parent:
    ROOT = ROOT.parent
B, R = ROOT/'benchmark', ROOT/'results'
assert (B/'case_scopes.csv').exists(); print('committed artifacts found (benchmark/, results/)')


committed artifacts found (benchmark/, results/)


## 1. FLARE dataset roles & provenance

Two genuine FLARE **data sources** plus one **download release** and a pending cohort:
- **FLARE22** organs (patient-level GT, via the FLARE 2024–2025 Task2 release) — the main benchmark hub.
- **FLARE23** class-14 tumor (real GT, **slice-level**, patient identity lost) — exploratory only.
- **FLARE24** — potential external cohort, **not used** until matched patient-level organ+tumor is verified.


In [2]:
roles = pd.DataFrame([
  ['FLARE22','organs (patient-level, no tumor)','patient','tests UNOBSERVED tumor + cross-source observability','3D organs (if NIfTI avail)'],
  ['FLARE23 (local)','class-14 tumor GT','slice (patient id lost)','EXPLORATORY tumor presence/area/relations','2D only'],
  ['FLARE24','organ + pan-cancer tasks','UNVERIFIED','only after matched patient-level audit','3D demo; eval after audit'],
  ['MSD Pancreas','pancreas + tumor','patient','pancreatic tumor presence/burden/containment','3D'],
  ['LiTS','liver + tumor','patient','liver tumor presence/burden/containment','3D'],
], columns=['dataset','labels','eval level','cancer-query use','visualization'])
print('Dataset roles (see results/audit/flare_provenance.md for full provenance):')
roles


Dataset roles (see results/audit/flare_provenance.md for full provenance):


,dataset,labels,eval level,cancer-query use,visualization
0,FLARE22,"organs (patient-level, no tumor)",patient,tests UNOBSERVED tumor + cross-source observability,3D organs (if NIfTI avail)
1,FLARE23 (local),class-14 tumor GT,slice (patient id lost),EXPLORATORY tumor presence/area/relations,2D only
2,FLARE24,organ + pan-cancer tasks,UNVERIFIED,only after matched patient-level audit,3D demo; eval after audit
3,MSD Pancreas,pancreas + tumor,patient,pancreatic tumor presence/burden/containment,3D
4,LiTS,liver + tumor,patient,liver tumor presence/burden/containment,3D


## 2. FLARE22 observability — pancreatic tumor stays UNOBSERVED

A FLARE22 case observes the pancreas **organ** but its source annotates **no tumor**, so
`pancreas_tumor_present` must be **Unknown** (not tumor-negative). Computed from the two
surfaced scopes — anatomy (`case_scopes`) ∩ annotation-capability (`annotation_scopes`) —
and the explicit `support_units`.


In [3]:
scopes = pd.read_csv(B/'case_scopes.csv')
ann = json.load(open(B/'annotation_scopes.json'))['annotation_capability_scopes']
units = json.load(open(B/'support_units.json'))['support_units']

def observe(case_row, feature):
    organs = set(str(case_row.observed_organs).split('|'))
    src = case_row.source_id; tumor_annot = set(ann[src]['annotates_tumor_for'])
    need = units[feature]
    anat_ok = all(o in organs for o in need['anatomy'])
    cap_ok = all((u=='tumor_annotation:*' and len(tumor_annot)>0) or
                 (u.startswith('tumor_annotation:') and u.split(':')[1] in tumor_annot)
                 for u in need['annotation_capability'])
    return 'OBSERVED (T/F)' if (anat_ok and cap_ok) else 'UNOBSERVED (U)'

msd = scopes[scopes.source_id=='msd_pancreas'].iloc[0]
fl  = scopes[scopes.source_id=='flare22'].iloc[0]
rows=[]
for feat in ['pancreas_present','pancreas_volume_cm3','pancreas_tumor_present']:
    rows.append([feat, observe(msd,feat), observe(fl,feat)])
print(f'MSD case {msd.case_id}  vs  FLARE case {fl.case_id}')
pd.DataFrame(rows, columns=['phenotype','MSD-Pancreas','FLARE22'])


MSD case pancreas_001  vs  FLARE case FLARE22_Tr_0001


,phenotype,MSD-Pancreas,FLARE22
0,pancreas_present,OBSERVED (T/F),OBSERVED (T/F)
1,pancreas_volume_cm3,OBSERVED (T/F),OBSERVED (T/F)
2,pancreas_tumor_present,OBSERVED (T/F),UNOBSERVED (U)


**Key:** `pancreas_tumor_present` is OBSERVED for MSD (can answer present/absent) but
**UNOBSERVED** for FLARE — so in an MSD↔FLARE pair it is *excluded* from comparison
(tumor-incomparable), never scored as tumor-negative. This is OAKG's observability
contribution, and it is exactly the annotation-capability ∩ anatomy support rule.


## 3. FLARE22's contribution to the main patient-level benchmark

FLARE22 is the **multi-organ hub** — without it γ is degenerate and the policy ablation,
native cross-dataset, and observation-boundary results are trivial or impossible.


In [4]:
ps = pd.read_csv(R/'tables'/'ranking_policy_selection.csv')
print('Policy ablation (needs FLARE22 multi-organ to be informative):')
print(ps.to_string(index=False))
nm = pd.read_csv(R/'native_cross_dataset'/'native_method_summary.csv')
fl = nm[(nm.stratum=='flare22-to-other-sources') & (nm.method.isin(['OAKG','OAKG-Union']))]
print('\nNative cross-dataset, FLARE22 as the bridge (flare22 -> other sources):')
print(fl[['stratum','method','mean','n_queries']].to_string(index=False))
ua = pd.read_csv(R/'union_ablation'/'union_ablation_paired_comparison.csv')
print('\nObservation-boundary ablation (uses FLARE22 one-sided coverage):')
print(ua[['regime','delta_obs','ci_low','ci_high','holm_p']].to_string(index=False))


Policy ablation (needs FLARE22 multi-organ to be informative):
                  method  mean_metric  served_rate  n_queries
OAKG-lexicographic [ref]     0.402694          1.0        111
      OAKG-product [ref]     0.402694          1.0        111
   OAKG-similarity [ref]     0.317940          1.0        111
    OAKG-threshold [ref]     0.316318          1.0        111

Native cross-dataset, FLARE22 as the bridge (flare22 -> other sources):
                 stratum     method     mean  n_queries
flare22-to-other-sources       OAKG 0.179272          8
flare22-to-other-sources OAKG-Union 0.000000          8

Observation-boundary ablation (uses FLARE22 one-sided coverage):
       regime  delta_obs    ci_low  ci_high  holm_p
   asymmetric   0.068095  0.022390 0.114561  0.0004
dataset_style   0.111879  0.044522 0.182173  0.0004
       random  -0.009730 -0.023495 0.000261  0.0004
      uniform   0.000769  0.000000 0.002307  1.0000


## 4. FLARE23 slice-level tumor stratum — EXPLORATORY (not confirmatory)

Real class-14 tumor GT, but **slice-level**: 364 slices are **not** 364 independent
patients, patient clustering is unrecoverable, and its **CIs/p-values are not
confirmatory**. Kept entirely separate from the 512-case patient benchmark.


In [5]:
tu = pd.read_csv(R/'strata'/'flare_tumor_realgt.csv')
print('FLARE23 cross-organ tumor stratum (EXPLORATORY, slice-level, paired deltas only):')
print(tu.to_string(index=False))
print('\nRead as: directional real-label corroboration (OAKG - zero-imp ~ +0.043), NOT a',
      'confirmatory patient-level result.')


FLARE23 cross-organ tumor stratum (EXPLORATORY, slice-level, paired deltas only):
   method  nDCG@10  delta_vs_ZeroImp  ci_low  ci_high  served_rate sig
       WL   0.6407            0.0677  0.0412   0.0945          1.0 SIG
     OAKG   0.6157            0.0427  0.0194   0.0668          1.0 SIG
  ZeroImp   0.5730            0.0000  0.0000   0.0000          1.0 NaN
  MissInd   0.5728           -0.0002 -0.0010   0.0005          1.0  ns
  MeanImp   0.5147           -0.0582 -0.0822  -0.0341          1.0 SIG
MaskedCos   0.1912           -0.3817 -0.4204  -0.3424          1.0 SIG

Read as: directional real-label corroboration (OAKG - zero-imp ~ +0.043), NOT a confirmatory patient-level result.


## 5. Cancer-query examples

The queries FLARE22 supports (its point is *not assuming* tumor absence):
- *"anatomically comparable but tumor-incomparable"* — FLARE cases match on organ
  morphology but their tumor status is Unknown, so they are excluded from tumor scoring.
- Patient-level cancer queries run on **MSD Pancreas** and **LiTS** (which annotate tumor);
  FLARE22 tests that unannotated tumor stays UNOBSERVED.

See `results/qualitative/` for runnable examples (e.g. Q1/Q6/Q8 pancreatic-tumor queries).


In [6]:
qr = pd.read_csv(R/'qualitative'/'qualitative_retrieval.csv')
tumor_qs = qr[qr.need.str.contains('tumour|tumor', case=False)][['query_id','need']].drop_duplicates()
print('Committed tumor-related qualitative queries (patient-level, MSD/LiTS):')
print(tumor_qs.to_string(index=False))


Committed tumor-related qualitative queries (patient-level, MSD/LiTS):
query_id                                                         need
      Q1            Pancreatic tumour, contained, in a large pancreas
      Q2                 Liver tumour, multifocal, high tumour burden
      Q4               Liver tumour with high burden in a large liver
      Q5                  Any tumour, multifocal, above-median burden
      Q6       Pancreatic tumour, contained, low burden (early-stage)
      Q7     Liver tumour, solitary lesion, above-median liver volume
      Q8 Invasive pancreatic tumour (present but NOT organ-contained)


## 6. FLARE24 — audit pending (not used)

FLARE24 has **separate** organ-segmentation and pan-cancer tasks, so a displayed organ
case may not have aligned tumor labels. Before any OAKG use, verify: which task the files
are from; same patient has both organ + tumor labels; matching case IDs / shapes / spacing
/ orientation / affine; tumor completeness; patient-level identity preserved. **Not merged
into the benchmark** until matched patient-level organ+tumor volumes are confirmed.

---
*This notebook consolidates the FLARE evaluations; provenance detail in
`results/audit/flare_provenance.md`, the observability bridge in
`results/audit/annotation_capability.md`.*
